# Social Media Impact Exploratory Data Analysis

## Dataset Overview

##### Who They Are?
- age : 13 - 19 -> teenagers
- gender : Male / Female
##### their social media behavior
- daily_social_media_hours : number of hours they spent on social media per day
- platform_usage : social media platform used
- screen_time_before_sleep : number of hours spent using a phone before bed
##### their wellbeing indicators
- sleep_hours : number of hours slept per night
- academic_performance : academic performance score (might be a GPA)
- physical_activity : daily hours spent doing physical activity (no screen time)
- social_interaction_level : in-person interaction (low - medium - high)
- stress_level : scale 1-10
- anxiety_level : scale 1-10
- addiction_level : social media addiction score, scale 1-10
- depression_label : target variable
    - 0 -> not depressed  
    - 1 -> depressed

## Importing Libraries

In [1]:
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression

## loading the data

In [2]:
df = pd.read_csv('data/Teen_Mental_Health_Dataset.csv')
df.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0


## EDA

In [3]:
df.shape

(1200, 13)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   object 
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   object 
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_level  1200 non-null   object 
 9   stress_level              1200 non-null   int64  
 10  anxiety_level             1200 non-null   int64  
 11  addiction_level           1200 non-null   int64  
 12  depression_label          1200 non-null   int64  
dtypes: float64(5), int64(5), object(3)
memory usage: 122.0+ KB


### Checking Null Values

In [5]:
df.isna().sum() ## no null values

age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64

### Checking Duplicate values

In [6]:
df.duplicated().sum() ## no duplicates

np.int64(0)

### Checking The Distribution

In [7]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,1200.0,NaN,NaN,NaN,15.928333,2.021947,13.0,14.0,16.0,18.0,19.0
gender,1200,2,male,615,NaN,NaN,NaN,NaN,NaN,NaN,NaN
daily_social_media_hours,1200.0,NaN,NaN,NaN,4.536667,2.029599,1.0,2.8,4.5,6.3,8.0
platform_usage,1200,3,Instagram,411,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sleep_hours,1200.0,NaN,NaN,NaN,6.449417,1.442677,4.0,5.2,6.5,7.6,9.0
screen_time_before_sleep,1200.0,NaN,NaN,NaN,1.740333,0.71666,0.5,1.1,1.8,2.4,3.0
academic_performance,1200.0,NaN,NaN,NaN,2.990383,0.576758,2.0,2.5,2.99,3.48,4.0
physical_activity,1200.0,NaN,NaN,NaN,1.0145,0.582185,0.0,0.5,1.0,1.5,2.0
social_interaction_level,1200,3,medium,416,NaN,NaN,NaN,NaN,NaN,NaN,NaN
stress_level,1200.0,NaN,NaN,NaN,5.445833,2.90329,1.0,3.0,5.0,8.0,10.0


- range validation
    - **age**: 13–19 → within expected teen range
    - **academic_performance**: 2.0–4.0 → consistent with GPA scale
    - **stress_level**, **anxiety_level**, **addiction_level**: 1–10 → within expected scale

## sanity check

In [8]:
check_cols = ["gender", "platform_usage",  "social_interaction_level", "depression_label"]
for col in check_cols:
    print(f"{col}: {df[col].unique()}")

    # there isn't any unexpected value

gender: ['male' 'female']
platform_usage: ['Instagram' 'TikTok' 'Both']
social_interaction_level: ['low' 'high' 'medium']
depression_label: [0 1]


## checking outliers

In [9]:
cols_to_check=["physical_activity", "screen_time_before_sleep", "sleep_hours", "daily_social_media_hours"]
for col in cols_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1-1.5*IQR
    upper_bound  = Q3+1.5*IQR

    outliers = (df[col]>upper_bound) | (df[col]<lower_bound)
    print(f"{col}: number of outliers={outliers.sum()}, lb={lower_bound} ub={upper_bound} ") 
    
    ## no outliers

physical_activity: number of outliers=0, lb=-1.0 ub=3.0 
screen_time_before_sleep: number of outliers=0, lb=-0.8499999999999996 ub=4.35 
sleep_hours: number of outliers=0, lb=1.600000000000001 ub=11.2 
daily_social_media_hours: number of outliers=0, lb=-2.45 ub=11.55 


## uni-variate analysis

### gender Column Distribution

In [10]:
df["gender"].value_counts(normalize=True)*100

gender
male      51.25
female    48.75
Name: proportion, dtype: float64

In [11]:
gender_counts = df['gender'].value_counts().reset_index()

fig = px.bar(gender_counts, x='gender', y='count', text='count', color='gender', title='gender Distribution')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='gender', yaxis_title='count', showlegend=False, height=500)
fig.show()

### platform_usage Column Distribution

In [12]:
df["platform_usage"].value_counts(normalize=True)*100

platform_usage
Instagram    34.250000
TikTok       33.166667
Both         32.583333
Name: proportion, dtype: float64

In [13]:
platform_counts = df['platform_usage'].value_counts().reset_index()

fig = px.bar(platform_counts, x='platform_usage', y='count', text='count', color='platform_usage', title='platform_usage Distribution')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='platform_usage', yaxis_title='count', showlegend=False, height=500)
fig.show()

### social_interaction_level Column Distribution

In [14]:
df["social_interaction_level"].value_counts(normalize=True)*100

social_interaction_level
medium    34.666667
low       34.583333
high      30.750000
Name: proportion, dtype: float64

In [15]:
soc_int_counts = df['social_interaction_level'].value_counts().reset_index()

fig = px.bar(soc_int_counts, x='social_interaction_level', y='count', text='count', color='social_interaction_level', title='social_interaction_level Distribution')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='social_interaction_level', yaxis_title='count', showlegend=False, height=500)
fig.show()

### stress_level Column Distribution

In [16]:
df["stress_level"].value_counts(normalize=True)*100

stress_level
4     11.583333
1     11.166667
5     10.750000
10    10.666667
9     10.166667
3      9.666667
6      9.500000
2      9.000000
7      8.833333
8      8.666667
Name: proportion, dtype: float64

In [17]:
stress_counts = df['stress_level'].value_counts().reset_index()

fig = px.bar(stress_counts, x='count', y='stress_level', text='count', color='stress_level',orientation='h', title='stress_level Distribution')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='count', yaxis_title='stress_level', showlegend=False, height=500, yaxis=dict(tickmode='linear', tick0=1, dtick=1))
fig.show()

### anxiety_level Column Distribution

In [18]:
df["anxiety_level"].value_counts(normalize=True)*100

anxiety_level
6     11.000000
4     10.916667
10    10.916667
8     10.833333
9     10.333333
3     10.250000
2      9.166667
7      9.083333
1      8.750000
5      8.750000
Name: proportion, dtype: float64

In [19]:
anxiety_counts = df['anxiety_level'].value_counts().reset_index()

fig = px.bar(anxiety_counts, x='count', y='anxiety_level', text='count', color='anxiety_level',orientation='h', title='anxiety_level Distribution')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='count', yaxis_title='anxiety_level', showlegend=False, height=500, yaxis=dict(tickmode='linear', tick0=1, dtick=1))
fig.show()

### addiction_level Column Distribution

In [20]:
df["addiction_level"].value_counts(normalize=True)*100

addiction_level
6     11.166667
8     11.000000
7     10.333333
2     10.000000
9      9.833333
4      9.750000
3      9.750000
10     9.583333
5      9.500000
1      9.083333
Name: proportion, dtype: float64

In [21]:
addiction_counts = df['addiction_level'].value_counts().reset_index()

fig = px.bar(addiction_counts, x='count', y='addiction_level', text='count', color='addiction_level',orientation='h', title='addiction_level Distribution')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='count', yaxis_title='addiction_level', showlegend=False, height=500, yaxis=dict(tickmode='linear', tick0=1, dtick=1))
fig.show()

### depression_label Column Distribution

In [22]:
df["depression_label"].value_counts(normalize=True)*100

depression_label
0    97.416667
1     2.583333
Name: proportion, dtype: float64

In [23]:
dep_counts = df['depression_label'].value_counts().reset_index()

fig = px.pie(dep_counts, names='depression_label', values='count', hole=0.4, title='depression_label Distribution', color='depression_label', )
fig.update_traces(textinfo='percent+label', pull=[0, 0.2])
fig.show()

### age Column Distribution

In [24]:
df["age"].unique() # discrete

array([14, 19, 17, 15, 18, 16, 13])

In [25]:
df["age"].value_counts(normalize=True)

age
13    0.166667
15    0.150000
18    0.143333
17    0.141667
16    0.135833
19    0.135000
14    0.127500
Name: proportion, dtype: float64

In [26]:
age_reset = df['age'].value_counts().reset_index()

fig = px.bar(age_reset, x='age', y='count', text='count', color='age', title='age Distribution')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='age', yaxis_title='count', showlegend=False, height=500)
fig.show()

### academic_performance Column Distribution

In [27]:
df["academic_performance"].describe()

count    1200.000000
mean        2.990383
std         0.576758
min         2.000000
25%         2.500000
50%         2.990000
75%         3.480000
max         4.000000
Name: academic_performance, dtype: float64

In [28]:
fig = px.histogram(df, x="academic_performance", nbins=50, marginal="box", title="academic_performance Distribution (raw)")
fig.add_vline(x=df["academic_performance"].mean(), line_dash='dash', line_color="brown", annotation_text=f"mean = {round(df['academic_performance'].mean(), 2)}", annotation_position="top right")
fig.add_vline(x=df["academic_performance"].median(), line_dash='dot', line_color="pink", annotation_text=f"median={round(df['academic_performance'].median(), 2)}", annotation_position="top left")
fig.update_layout(xaxis_title='academic_performance', yaxis_title='Frequency', height=600, width=1400)
fig.update_traces(selector=dict(type='box'), notched=False)
fig.show()

### daily_social_media_hours Column Distribution

In [29]:
df["daily_social_media_hours"].describe()

count    1200.000000
mean        4.536667
std         2.029599
min         1.000000
25%         2.800000
50%         4.500000
75%         6.300000
max         8.000000
Name: daily_social_media_hours, dtype: float64

In [30]:
fig = px.histogram(df, x="daily_social_media_hours", nbins=40, marginal="box", title="daily_social_media_hours Distribution")
fig.add_vline(x=df["daily_social_media_hours"].mean(), line_dash='dash', line_color="brown", annotation_text=f"mean = {round(df['daily_social_media_hours'].mean(), 2)}", annotation_position="top right")
fig.add_vline(x=df["daily_social_media_hours"].median(), line_dash='dot', line_color="pink", annotation_text=f"median={round(df['daily_social_media_hours'].median(), 2)}", annotation_position="top left")
fig.update_layout(xaxis_title='daily_social_media_hours', yaxis_title='Frequency', height=600, width=1400)
fig.update_traces(selector=dict(type='box'), notched=False)
fig.show()

### sleep_hours Column Distribution

In [31]:
df["sleep_hours"].describe()

count    1200.000000
mean        6.449417
std         1.442677
min         4.000000
25%         5.200000
50%         6.500000
75%         7.600000
max         9.000000
Name: sleep_hours, dtype: float64

In [32]:
fig = px.histogram(df, x="sleep_hours", nbins=30, marginal="box", title="sleep_hours Distribution")
fig.add_vline(x=df["sleep_hours"].mean(), line_dash='dash', line_color="brown", annotation_text=f"mean = {round(df['sleep_hours'].mean(), 2)}", annotation_position="top right")
fig.add_vline(x=df["sleep_hours"].median(), line_dash='dot', line_color="pink", annotation_text=f"median={round(df['sleep_hours'].median(), 2)}", annotation_position="top left")
fig.update_layout(xaxis_title='sleep_hours', yaxis_title='Frequency', height=600, width=1400)
fig.update_traces(selector=dict(type='box'), notched=False)
fig.show()

### screen_time_before_sleep Column Distribution

In [33]:
df["screen_time_before_sleep"].describe()

count    1200.000000
mean        1.740333
std         0.716660
min         0.500000
25%         1.100000
50%         1.800000
75%         2.400000
max         3.000000
Name: screen_time_before_sleep, dtype: float64

In [34]:
n_bins = df["screen_time_before_sleep"].nunique()

fig = px.histogram(df, x="screen_time_before_sleep", nbins=n_bins, marginal="box", title="screen_time_before_sleep Distribution")
fig.add_vline(x=df["screen_time_before_sleep"].mean(), line_dash='dash', line_color="brown", annotation_text=f"mean = {round(df['screen_time_before_sleep'].mean(), 2)}", annotation_position="top right")
fig.add_vline(x=df["screen_time_before_sleep"].median(), line_dash='dot', line_color="pink", annotation_text=f"median={round(df['screen_time_before_sleep'].median(), 2)}", annotation_position="top left")
fig.update_layout(xaxis_title='screen_time_before_sleep', yaxis_title='Frequency', height=600, width=1400)
fig.update_traces(selector=dict(type='box'), notched=False)
fig.show()

### physical_activity Column Distribution

In [35]:
df["physical_activity"].describe()

count    1200.000000
mean        1.014500
std         0.582185
min         0.000000
25%         0.500000
50%         1.000000
75%         1.500000
max         2.000000
Name: physical_activity, dtype: float64

In [36]:
n_bins = df["physical_activity"].nunique()

fig = px.histogram(df, x="physical_activity", nbins= n_bins, marginal="box", title="physical_activity Distribution")
fig.add_vline(x=df["physical_activity"].mean(), line_dash='dash', line_color="brown", annotation_text=f"mean = {round(df['physical_activity'].mean(), 2)}", annotation_position="top right")
fig.add_vline(x=df["physical_activity"].median(), line_dash='dot', line_color="pink", annotation_text=f"median={round(df['physical_activity'].median(), 2)}", annotation_position="top left")
fig.update_layout(xaxis_title='physical_activity', yaxis_title='Frequency', height=600, width=1400)
fig.show()

## bi-variate analysis

### depression_label vs age

In [37]:
df.groupby(["age"])["depression_label"].value_counts(normalize=True)*100

age  depression_label
13   0                   98.000000
     1                    2.000000
14   0                   98.039216
     1                    1.960784
15   0                   95.555556
     1                    4.444444
16   0                   97.546012
     1                    2.453988
17   0                   98.823529
     1                    1.176471
18   0                   97.674419
     1                    2.325581
19   0                   96.296296
     1                    3.703704
Name: proportion, dtype: float64

In [38]:
age_dep = df.groupby(["age"])["depression_label"].value_counts().reset_index()
age_dep['depression_label'] = age_dep['depression_label'].astype(str)

fig = px.bar(age_dep, x='age', y='count', color='depression_label', barmode='group', text='count', title='depression_label by age')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='age', yaxis_title='Number of teens', legend_title='depression_label', height=500)
fig.show()

### depression_label vs addiction_level

In [39]:
df.groupby(["addiction_level"])["depression_label"].value_counts(normalize=True)*100

addiction_level  depression_label
1                0                   97.247706
                 1                    2.752294
2                0                   99.166667
                 1                    0.833333
3                0                   94.017094
                 1                    5.982906
4                0                   95.726496
                 1                    4.273504
5                0                   97.368421
                 1                    2.631579
6                0                   98.507463
                 1                    1.492537
7                0                   99.193548
                 1                    0.806452
8                0                   99.242424
                 1                    0.757576
9                0                   97.457627
                 1                    2.542373
10               0                   95.652174
                 1                    4.347826
Name: proportion, dtype: f

In [40]:
addiction_level_dep = df.groupby(["addiction_level"])["depression_label"].value_counts().reset_index()
addiction_level_dep['depression_label'] = addiction_level_dep['depression_label'].astype(str)

fig = px.bar(addiction_level_dep, x='addiction_level', y='count', color='depression_label', barmode='group', text='count', title='depression_label by addiction_level')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='addiction_level', yaxis_title='Number of teens', legend_title='depression_label', height=500)
fig.show()

### depression_label vs anxiety_level

In [41]:
df.groupby(["anxiety_level"])["depression_label"].value_counts(normalize=True)*100

anxiety_level  depression_label
1              0                   100.000000
2              0                   100.000000
3              0                   100.000000
4              0                   100.000000
5              0                   100.000000
6              0                   100.000000
7              0                    94.495413
               1                     5.504587
8              0                    94.615385
               1                     5.384615
9              0                    91.129032
               1                     8.870968
10             0                    94.656489
               1                     5.343511
Name: proportion, dtype: float64

In [42]:
anxiety_level_dep = df.groupby(["anxiety_level"])["depression_label"].value_counts().reset_index()
anxiety_level_dep['depression_label'] = anxiety_level_dep['depression_label'].astype(str)

fig = px.bar(anxiety_level_dep, x='anxiety_level', y='count', color='depression_label', barmode='group', text='count', title='depression_label by anxiety_level')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='anxiety_level', yaxis_title='Number of teens', legend_title='depression_label', height=500)
fig.show()

### depression_label vs stress_level

In [43]:
df.groupby(["stress_level"])["depression_label"].value_counts(normalize=True)*100

stress_level  depression_label
1             0                   100.000000
2             0                   100.000000
3             0                   100.000000
4             0                   100.000000
5             0                   100.000000
6             0                   100.000000
7             0                    91.509434
              1                     8.490566
8             0                    93.269231
              1                     6.730769
9             0                    95.081967
              1                     4.918033
10            0                    92.968750
              1                     7.031250
Name: proportion, dtype: float64

In [44]:
stress_level_dep = df.groupby(["stress_level"])["depression_label"].value_counts().reset_index()
stress_level_dep['depression_label'] = stress_level_dep['depression_label'].astype(str)

fig = px.bar(stress_level_dep, x='stress_level', y='count', color='depression_label', barmode='group', text='count', title='depression_label by stress_level')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='stress_level', yaxis_title='Number of teens', legend_title='depression_label', height=500)
fig.show()

### depression_label vs social_interaction_level

In [45]:
df.groupby(["social_interaction_level"])["depression_label"].value_counts(normalize=True)*100

social_interaction_level  depression_label
high                      0                   97.289973
                          1                    2.710027
low                       0                   97.831325
                          1                    2.168675
medium                    0                   97.115385
                          1                    2.884615
Name: proportion, dtype: float64

In [46]:
social_interaction_level_dep = df.groupby(["social_interaction_level"])["depression_label"].value_counts().reset_index()
social_interaction_level_dep['depression_label'] = social_interaction_level_dep['depression_label'].astype(str)

fig = px.bar(social_interaction_level_dep, x='social_interaction_level', y='count', color='depression_label', barmode='group', text='count', title='depression_label by social_interaction_level')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='social_interaction_level', yaxis_title='Number of teens', legend_title='depression_label', height=500)
fig.show()

### depression_label vs platform_usage

In [47]:
df.groupby(["platform_usage"])["depression_label"].value_counts(normalize=True)*100

platform_usage  depression_label
Both            0                   97.698210
                1                    2.301790
Instagram       0                   97.566910
                1                    2.433090
TikTok          0                   96.984925
                1                    3.015075
Name: proportion, dtype: float64

In [48]:
platform_usage_dep = df.groupby(["platform_usage"])["depression_label"].value_counts().reset_index()
platform_usage_dep['depression_label'] = platform_usage_dep['depression_label'].astype(str)

fig = px.bar(platform_usage_dep, x='platform_usage', y='count', color='depression_label', barmode='group', text='count', title='depression_label by platform_usage')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='platform_usage', yaxis_title='Number of teens', legend_title='depression_label', height=500)
fig.show()

### depression_label vs gender

In [49]:
df.groupby(["gender"])["depression_label"].value_counts(normalize=True)*100

gender  depression_label
female  0                   97.094017
        1                    2.905983
male    0                   97.723577
        1                    2.276423
Name: proportion, dtype: float64

In [50]:
gender_dep = df.groupby(["gender"])["depression_label"].value_counts().reset_index()
gender_dep['depression_label'] = gender_dep['depression_label'].astype(str)

fig = px.bar(gender_dep, x='gender', y='count', color='depression_label', barmode='group', text='count', title='depression_label by gender')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='gender', yaxis_title='Number of teens', legend_title='depression_label', height=500)
fig.show()

### depression_label vs academic_performance

In [51]:
df.groupby(["depression_label"])["academic_performance"].mean()

depression_label
0    2.990248
1    2.995484
Name: academic_performance, dtype: float64

In [52]:
fig = px.box(df, x="depression_label", y="academic_performance", color="depression_label", title="depression_label by academic_performance")
fig.update_layout(xaxis_title="depression_label", yaxis_title="academic_performance", showlegend=False)
fig.show()

### depression_label vs daily_social_media_hours

In [53]:
df.groupby(["depression_label"])["daily_social_media_hours"].mean()

depression_label
0    4.478785
1    6.719355
Name: daily_social_media_hours, dtype: float64

In [54]:
fig = px.box(df, x="depression_label", y="daily_social_media_hours", color="depression_label", title="depression_label by daily_social_media_hours")
fig.update_layout(xaxis_title="depression_label", yaxis_title="daily_social_media_hours", showlegend=False)
fig.show()

### depression_label vs sleep_hours

In [55]:
df.groupby(["depression_label"])["sleep_hours"].mean()

depression_label
0    6.494183
1    4.761290
Name: sleep_hours, dtype: float64

In [56]:
fig = px.box(df, x="depression_label", y="sleep_hours", color="depression_label", title="depression_label by sleep_hours")
fig.update_layout(xaxis_title="depression_label", yaxis_title="sleep_hours", showlegend=False)
fig.show()

### depression_label vs screen_time_before_sleep

In [57]:
df.groupby(["depression_label"])["screen_time_before_sleep"].mean()

depression_label
0    1.742258
1    1.667742
Name: screen_time_before_sleep, dtype: float64

In [58]:
fig = px.box(df, x="depression_label", y="screen_time_before_sleep", color="depression_label", title="depression_label by screen_time_before_sleep")
fig.update_layout(xaxis_title="depression_label", yaxis_title="screen_time_before_sleep", showlegend=False)
fig.show()

### depression_label vs physical_activity

In [59]:
df.groupby(["depression_label"])["physical_activity"].mean()

depression_label
0    1.016168
1    0.951613
Name: physical_activity, dtype: float64

In [60]:
fig = px.box(df, x="depression_label", y="physical_activity", color="depression_label", title="depression_label by physical_activity")
fig.update_layout(xaxis_title="depression_label", yaxis_title="physical_activity", showlegend=False)
fig.show()

### addiction_level vs daily_social_media_hours

In [61]:
df.groupby(["addiction_level"])["daily_social_media_hours"].mean()

addiction_level
1     4.799083
2     4.355000
3     4.573504
4     4.537607
5     4.637719
6     4.470896
7     4.503226
8     4.667424
9     4.652542
10    4.182609
Name: daily_social_media_hours, dtype: float64

In [62]:
fig = px.box(df, x="addiction_level", y="daily_social_media_hours", color="addiction_level", title="addiction_level by daily_social_media_hours")
fig.update_layout(xaxis_title="addiction_level", yaxis_title="daily_social_media_hours", showlegend=False)
fig.show()

### social_interaction_level vs daily_social_media_hours

In [63]:
df.groupby(["social_interaction_level"])["daily_social_media_hours"].mean()

social_interaction_level
high      4.457182
low       4.539518
medium    4.604327
Name: daily_social_media_hours, dtype: float64

In [64]:
fig = px.box(df, x="social_interaction_level", y="daily_social_media_hours", color="social_interaction_level", title="social_interaction_level by daily_social_media_hours")
fig.update_layout(xaxis_title="social_interaction_level", yaxis_title="daily_social_media_hours", showlegend=False)
fig.show()

### anxiety_level vs stress_level

In [65]:
df.groupby(["anxiety_level"])["stress_level"].value_counts(normalize=True)*100

anxiety_level  stress_level
1              3               17.142857
               4               14.285714
               10              13.333333
               1               11.428571
               2               10.476190
                                 ...    
10             3                9.160305
               4                9.160305
               8                6.870229
               2                4.580153
               7                4.580153
Name: proportion, Length: 100, dtype: float64

In [66]:
anxiety_level_dep = df.groupby(["anxiety_level"])["stress_level"].value_counts().reset_index()
anxiety_level_dep['stress_level'] = anxiety_level_dep['stress_level'].astype(str)
anxiety_level_dep['anxiety_level'] = anxiety_level_dep['anxiety_level'].astype(str)

fig = px.bar(anxiety_level_dep, x='anxiety_level', y='count', color='stress_level', barmode='group', text='count', title='stress_level by anxiety_level')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='anxiety_level', yaxis_title='Number of teens', legend_title='stress_level', height=500)
fig.show()

### sleep_hours vs screen_time_before_sleep

In [67]:
df[["sleep_hours", "screen_time_before_sleep"]].corr()

,sleep_hours,screen_time_before_sleep
sleep_hours,1.000000,0.010235
screen_time_before_sleep,0.010235,1.000000


In [68]:
fig = px.scatter(df, x='sleep_hours', y='screen_time_before_sleep', title="sleep_hours vs screen_time_before_sleep")
fig.update_layout(xaxis_title='sleep_hours', yaxis_title='screen_time_before_sleep')
fig.show()

### sleep_hours vs daily_social_media_hours

In [69]:
df[["sleep_hours", "daily_social_media_hours"]].corr()

,sleep_hours,daily_social_media_hours
sleep_hours,1.000000,-0.009472
daily_social_media_hours,-0.009472,1.000000


In [70]:
fig = px.scatter(df, x='sleep_hours', y='daily_social_media_hours', title="sleep_hours vs daily_social_media_hours")
fig.update_layout(xaxis_title='sleep_hours', yaxis_title='daily_social_media_hours')
fig.show()

## multi-variate analysis

### depression_label vs stress_level vs anxiety_level

In [71]:
fig = px.scatter(df, x='stress_level', y='anxiety_level', color='depression_label', title='stress vs anxiety colored by depression')
fig.show()

### depression_label vs stress_level vs sleep_hours

In [72]:
fig = px.scatter(df, x='sleep_hours', y='stress_level', color='depression_label', size='anxiety_level')
fig.show()

### depression_label vs daily_social_media_hours vs social_interaction_level

In [73]:
fig = px.box(df, x='social_interaction_level', y='daily_social_media_hours', color='depression_label')
fig.show()

### depression_label vs daily_social_media_hours vs sleep_hours

In [74]:
fig = px.scatter(df, x='daily_social_media_hours', y='sleep_hours', color='depression_label', trendline='ols')
fig.show()

## key findings

- target variable (depression_label) is highly imbalanced (only 2.6% positive cases)

- daily social media usage and sleep hours show the strongest individual relationship with depression
`depressed students spend more time on social media and sleep considerably fewer hours on average`

- stress level and anxiety level are also strong indicators of depression
`most depressed cases are concentrated at high stress and high anxiety levels`

- gender, age, academic performance, platform usage, social interaction level, physical activity, screen time before sleep, and addiction level variables show no meaningful relationship with the target

- correlation analysis reveals almost no linear relationship between daily social media hours and sleep hours or between sleep hours and screen time before sleep despite some of these variables being individually associated with depression

- multivariate analysis confirms that the clearest separation between depressed and non depressed students occurs when combining high stress, high anxiety, low sleep duration, and high daily social media usage

- Overall, the most informative features in this dataset are:
  - `stress_level`
  - `anxiety_level`
  - `sleep_hours`
  - `daily_social_media_hours`

## preprocessing

In [75]:
df.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0


In [76]:
X = df.drop("depression_label", axis=1)
y = df["depression_label"]

In [77]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, shuffle=True, random_state=42)

In [78]:
cols = ["gender", "social_interaction_level"]

for col in cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

In [79]:
X_train = pd.get_dummies(X_train, columns=["platform_usage"], dtype=int)

X_test = pd.get_dummies(X_test, columns=["platform_usage"], dtype=int)

In [80]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [81]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [82]:
y_pred_lr = lr.predict(X_test)

In [83]:
print(accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))

0.9875
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       234
           1       1.00      0.50      0.67         6

    accuracy                           0.99       240
   macro avg       0.99      0.75      0.83       240
weighted avg       0.99      0.99      0.99       240

[[234   0]
 [  3   3]]
